# Mini Project: Sentiment Assistant with BERT Fine-Tuning

Welcome to the final day mini project!
In this lab you will fine-tune `bert-base-uncased` on movie reviews, evaluate the model, and connect the dots with real customer-support scenarios.

## Prerequisites & Story Setup

- **Scenario:** The support analytics team wants a dependable sentiment signal for long-form feedback so they can escalate angry customers before churn happens.
- **Environment needs:** Python 3.9+, a GPU-enabled runtime (Colab, Kaggle, or a local GPU machine), ~6 GB free VRAM.
- **Packages:** `torch`, `transformers`, `datasets`, `accelerate`, `evaluate`.

In [5]:
!pip install -q torch transformers datasets accelerate evaluate

## Imports & Hardware Check

We always begin by confirming versions and hardware. If you see `GPU available: False`, switch the runtime (Runtime → Change runtime type → GPU).

In [6]:
import platform
import torch
from transformers import BertTokenizer, BertForSequenceClassification

print("Python version      :", platform.python_version())
print("PyTorch version     :", torch.__version__)
print("GPU available       :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Python version      : 3.13.16
PyTorch version     : 2.14.1+cu130
GPU available       : True


## Load the IMDB Reviews Dataset

We rely on IMDB because it is balanced (25k positive / 25k negative) and already split.

In [8]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


💬 Hugging Face `datasets` returns a `DatasetDict` with `train` and `test` splits. Each row is a `(text, label)` pair, exactly what our model expects.

Take a quick peek at samples to make sentiment concrete:

In [ ]:
for sample in dataset["train"].shuffle(seed=42).select(range(2)):
    print("Label:", "Positive" if sample["label"] else "Negative")
    print(sample["text"][:250], "...\n")

## Tokenizer Setup & Data Pipeline

BERT uses WordPiece tokenization to handle rare or unknown words by breaking them into subword units. It adds `[CLS]` and `[SEP]` tokens, and attention masks tell the model which tokens are real versus padding.

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

🧠 we import this tokenizer to reuse the same vocabulary the base model learned in 2018

Next, convert raw text → token IDs, attention masks, and segment IDs (token type IDs).

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

encoded = dataset.map(tokenize_batch, batched=True)
encoded = encoded.rename_column("label", "labels")
encoded.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"],
)

train_ds = encoded["train"]
test_ds  = encoded["test"]

🗒️ `datasets.map(batched=True)` keeps Hugging Face tokenization inside an efficient, cached pipeline, so we don't manually juggle NumPy arrays. `set_format("torch")` hands us tensors ready for training.

## Initialize the Fine-Tuning Model

Now we load `BertForSequenceClassification`, which already bundles the encoder + classification head.

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
)
model.to(device)
model.config

💡 we import this model to reuse the 110M parameters learned on BooksCorpus + Wikipedia. We only fine-tune for a couple of epochs, which is why learning rates in the 2e-5 range are standard.

## Train and Monitor

On a T4 GPU (the one you find in Google Colab), two epochs take ~15 minutes. Watch both training and validation accuracy. The Hugging Face `Trainer` sets up the AdamW optimizer, the loss, and the accuracy metric for us.

**#To-Do:** complete the `Trainer(...)` call.

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Using 1 epoch for demonstration to ensure faster training inside free tier limits
EPOCHS = 1

training_args = TrainingArguments(
    output_dir="./bert-imdb",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

history = trainer.train()

📈 Point out how the validation accuracy plateau indicates when to stop. Encourage screenshots of the learning curves for portfolios.

## Evaluate on the Held-Out Test Set

Even though the `Trainer` already reports validation metrics, we re-run evaluation on the untouched test pipeline to mimic production QA.

**#To-Do:** evaluate on the test set.

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

✅ Here you will notice whether accuracy crosses the ~0.90 classroom benchmark.

**#To-Do:** Use this to discuss acceptable error rates for real support teams.

## Build a Reusable Inference Helper

Wrap everything in a function so that you can paste real support transcripts and get an instant score.

**#To-Do:** complete `predict_sentiment`.

### Discussion: Acceptable Error Rates for Real Support Teams

When deploying a sentiment classifier (like this BERT model) in a production customer support queue:
- **Acceptable Error Rates:** While a classroom setting targets `>90%` accuracy, a production support pipeline needs to consider **precision vs. recall**. If the goal is to prevent churn by escalating angry customers, a high recall for negative sentiment (e.g., `>95%` of truly negative cases caught) is vital, even if it leads to lower precision (some false positives).
- **Impact of Errors:** A false positive (marking a neutral/happy customer as angry) is relatively cheap—an agent reviews it and moves on. A false negative (missing a genuinely angry customer) means a missed escalation, which directly impacts customer churn.

In [ ]:
def predict_sentiment(text: str):
    model.eval()
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    pred_label_idx = np.argmax(probs)
    label = "Positive" if pred_label_idx == 1 else "Negative"
    return label, float(probs[pred_label_idx])

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

For the previous example, you should see `Prediction: Positive (confidence=0.5) slightly more or less`.

🧭 confidence scores are essential when deciding whether to auto-respond or escalate to a human.

## Reflection & Next Steps

1. **Why fine-tuning matters:** You reused a public checkpoint to hit >90% accuracy with minimal data.
2. **Transferable skills:** Everything here also applies to classification tasks in HR, legal, or product analytics.
3. **What you can do with this:** domain adaptation, multilingual checkpoints (DistilBERT multilingual, XLM-R), and monitoring (log data drift, build dashboards).

**#To-Do:** Answer the following reflection questions in markdown cells:

- What lever (data cleaning, hyperparameters, more epochs) most improved results?
- Where would you add guardrails before deploying this sentiment signal live?
- Which stakeholders benefit the most (support lead, product manager, compliance officer)?

### Reflections & Answers

- **What lever most improved results?**
  Transitioning from a basic TF-IDF model to a pre-trained **Transformer (BERT)** provides the largest leap in contextual understanding. Beyond that, adjusting the **learning rate** (keeping it small, around `2e-5`) and **fine-tuning for 1 to 2 epochs** prevents catastrophic forgetting while adapting the weights to movie reviews.

- **Where would you add guardrails before deploying this sentiment signal live?**
  1. **Confidence Thresholding:** Only auto-escalate or label when the model's confidence is above `85%`. For scores between `50%-85%`, route them to a secondary standard review queue.
  2. **Input Length & Out-of-Distribution Checks:** Flag inputs that are extremely short (single words) or completely different from the training data (e.g., strings containing only code or garbage characters).
  3. **Bias & Fair Evaluation Monitoring:** Periodically audit predictions manually to ensure language patterns or demographic indicators do not receive systematically skewed sentiment evaluations.

- **Which stakeholders benefit the most?**
  1. **Support Lead:** Gets real-time triage and prioritization, allowing teams to focus on high-priority frustrated customers.
  2. **Product Manager:** Gains structured, aggregated feedback trends pointing out feature friction or bugs over time.
  3. **Compliance & QA Officers:** Can quickly audit negative interactions to maintain service level agreements and spot training gaps.